In [1]:
import numpy as np
import numba as nb
import os

In [2]:
# FOR TESTING #

@nb.njit(parallel=True, fastmath=True, nogil=True)
def div_debug(U, P, neighbors, points):
    divU = np.zeros(U.shape[:-1])
    for p in nb.prange(points):
        
        xup, xdn, yup, ydn, zup, zdn = neighbors[p]
        divU[p] = ( U[xup, 0] - U[xdn, 0] +
                    U[yup, 1] - U[ydn, 1] +
                    U[zup, 2] - U[zdn, 2] ) * 0.5
        
    return np.mean(divU), np.max(divU), np.mean(P), np.mean(U), np.max(U)

The nematic order, $Q_{ij}$, is given by

$$Q_{ij} = S(n_i n_j - \frac{1}{3}).$$

$n_i$ is a unit vector. Thus, $Q_{ij}$ is traceless and symmetric, giving it 5 degrees of freedom,

$$Q_{ij} = \begin{pmatrix}
Q_{xx} & Q_{xy} & Q_{xz} \\
Q_{xy} & Q_{yy} & Q_{yz} \\
Q_{xz} & Q_{yz} & -Q_{xx} - Q_{yy}
\end{pmatrix}.$$

$$Q_{xx} = n_x^2 - 1/3,$$
$$Q_{xy} = n_x n_y,$$
$$Q_{xz} = n_x n_z,$$
$$Q_{yy} = n_y^2 - 1/3,$$
$$Q_{yz} = n_y n_z.$$

The time evolution of $Q_{ij}$ is given by

$$\partial_t Q_{ij} = -U_k \partial_k Q_{ij} + \lambda E_{ij} + [Q, \omega] + \Gamma H_{ij}$$

$$ H_{ij} = - \frac{\delta F_{\mathrm{LdG}}}{\partial Q_{ij}} + (\mathbb{I}/3)\mathrm{Tr}\left[\frac{\delta F_{\mathrm{LdG}}}{\partial Q_{ij}}\right] = - Q_{ij} (A + C \mathrm{Tr}[Q^2]) - B(Q^2_{ij} - \mathrm{Tr}[Q^2](\mathbb{I}/3)) + K \partial_k^2 Q_{ij}$$

From its form, we see $H_{ij}$ is also traceless and symmetric.

$$\mathrm{Tr}[Q^2] = \mathrm{Tr}\left[ \begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix}\right] = \\ \mathrm{Tr}\left[\begin{pmatrix}\left(Q_{xx}\right)^2+\left(Q_{xy}\right)^2+\left(Q_{xz}\right)^2&Q_{xx}Q_{xy}+Q_{xy}Q_{yy}+Q_{xz}Q_{yz}&-Q_{xz}Q_{yy}+Q_{xy}Q_{yz}\\ Q_{xy}Q_{xx}+Q_{yy}Q_{xy}+Q_{yz}Q_{xz}&\left(Q_{xy}\right)^2+\left(Q_{yy}\right)^2+\left(Q_{yz}\right)^2&-Q_{yz}Q_{xx}+Q_{xy}Q_{xz}\\ -Q_{xz}Q_{yy}+Q_{yz}Q_{xy}&-Q_{yz}Q_{xx}+Q_{xz}Q_{xy}&\left(Q_{xz}\right)^2+\left(Q_{yz}\right)^2+\left(-Q_{xx}-Q_{yy}\right)^2\end{pmatrix}\right] = 2\left(Q_{xx}\right)^2+2\left(Q_{xy}\right)^2+2\left(Q_{xz}\right)^2+2\left(Q_{yy}\right)^2+2\left(Q_{yz}\right)^2+2Q_{xx}Q_{yy}.$$

Strain rate and vorticity tensors are given by

$$E_{ij} = \frac{1}{2}[\partial_i u_j + \partial_j u_i]$$

and 

$$\omega_{ij} = \frac{1}{2}[\partial_i u_j - \partial_j u_i]$$

$E_{ij}$ is symmetric and $\omega_{ij}$ is antisymmetric. Incompressibility imposes that $E_{ij}$ is traceless.

Thus strain rate has 5 degrees of freedom,

$$E_{ij} = \begin{pmatrix}
E_{xx} & E_{xy} & E_{xz} \\
E_{xy} & E_{yy} & E_{yz} \\
E_{xz} & E_{yz} & -E_{xx} - E_{yy}
\end{pmatrix}.$$

$$E_{xx} = \partial_x U_x,$$
$$E_{xy} = (\partial_x U_y + \partial_y U_x)/2,$$
$$E_{xz} = (\partial_x U_z + \partial_z U_x)/2,$$
$$E_{yy} = \partial_y U_y,$$
$$E_{yz} = (\partial_y U_z + \partial_z U_y)/2.$$

Vorticity only has 3 degrees of freedom,

$$\omega_{ij} = \begin{pmatrix}
0 & \omega_{xy} & \omega_{xz} \\
-\omega_{xy} & 0 & \omega_{yz} \\
-\omega_{xz} & -\omega_{yz} & 0
\end{pmatrix}.$$

$$\omega_{xy} = (\partial_x U_y - \partial_y U_x)/2,$$
$$\omega_{xz} = (\partial_x U_z - \partial_z U_x)/2,$$
$$\omega_{yz} = (\partial_y U_z - \partial_z U_y)/2.$$

$$[Q, \omega] = \begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix}\begin{pmatrix}0&ω_{xy}&ω_{xz}\\ -ω_{xy}&0&ω_{yz}\\ -ω_{xz}&-ω_{yz}&0\end{pmatrix}-\begin{pmatrix}0&ω_{xy}&ω_{xz}\\ -ω_{xy}&0&ω_{yz}\\ -ω_{xz}&-ω_{yz}&0\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix} = $$
$$\begin{pmatrix}-2ω_{xy}Q_{xy}-2ω_{xz}Q_{xz}&ω_{xy}Q_{xx}-ω_{yz}Q_{xz}-ω_{xy}Q_{yy}-ω_{xz}Q_{yz}&2ω_{xz}Q_{xx}+ω_{yz}Q_{xy}+ω_{xz}Q_{yy}-ω_{xy}Q_{yz}\\ -ω_{xy}Q_{yy}-ω_{xz}Q_{yz}-ω_{yz}Q_{xz}+ω_{xy}Q_{xx}&2ω_{xy}Q_{xy}-2ω_{yz}Q_{yz}&2ω_{yz}Q_{yy}+ω_{yz}Q_{xx}+ω_{xz}Q_{xy}+ω_{xy}Q_{xz}\\ 2ω_{xz}Q_{xx}+ω_{yz}Q_{xy}+ω_{xz}Q_{yy}-ω_{xy}Q_{yz}&2ω_{yz}Q_{yy}+ω_{yz}Q_{xx}+ω_{xz}Q_{xy}+ω_{xy}Q_{xz}&2ω_{xz}Q_{xz}+2ω_{yz}Q_{yz}\end{pmatrix}$$

Which is traceless and symmetric.

The time evolution of the flow field $U_i$ is given by

$$\partial_t U_i = -U_k \partial_k U_i + \nu \partial_j^2 U_i + \partial_j \Pi_{ij} - \partial_i p,\\ \partial_i U_i = 0.$$
$$\Pi_{ij} = -\lambda H_{ij} - \zeta Q_{ij} + [Q, H].$$

We see that $-\lambda H_{ij} - \zeta Q_{ij}$ is traceless and symmetric with 5 degrees of freedom. $\Pi^S_{ij} =-\lambda H_{ij} - \zeta Q_{ij} $.

$$[Q, H] = \begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix}\begin{pmatrix}H_{xx}&H_{xy}&H_{xz}\\ H_{xy}&H_{yy}&H_{yz}\\  H_{xz}&H_{yz}&-H_{xx}-H_{yy}\end{pmatrix}-\begin{pmatrix}H_{xx}&H_{xy}&H_{xz}\\H_{xy}&H_{yy}&H_{yz}\\ H_{xz}&H_{yz}&-H_{xx}-H_{yy}\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}&Q_{xz}\\ Q_{xy}&Q_{yy}&Q_{yz}\\ Q_{xz}&Q_{yz}&-Q_{xx}-Q_{yy}\end{pmatrix}= $$
$$\begin{pmatrix}0&Q_{xx}H_{xy} + Q_{xy}H_{yy} + Q_{xz}H_{yz} - Q_{xy}H_{xx} - Q_{yy}H_{xy} - Q_{yz}H_{xz}&2 Q_{xx} H_{xz} + Q_{yy} H_{xz} - Q_{yz} H_{yy} + Q_{xy} H_{yz} - 2 Q_{xz} H_{xx} - Q_{yz} H_{xy}\\ Q_{xy}H_{xx} + Q_{yy}H_{xy} + Q_{yz}H_{xz} -Q_{xx}H_{xy} - Q_{xy}H_{yy} - Q_{xz}H_{yz} &0&-2Q_{xz}H_{yy}+2Q_{yy}H_{yz}+Q_{xx}H_{yz}-Q_{yz}H_{xx}-Q_{xz}H_{xy}+Q_{xy}H_{xz}\\ -2 Q_{xx} H_{xz} - Q_{yy} H_{xz} + Q_{yz} H_{yy} - Q_{xy} H_{yz} + 2 Q_{xz} H_{xx} + Q_{yz} H_{xy}&2Q_{xz}H_{yy}-2Q_{yy}H_{yz}-Q_{xx}H_{yz}+Q_{yz}H_{xx}+Q_{xz}H_{xy}-Q_{xy}H_{xz}&0\end{pmatrix}$$

Which is anti-symmetric with 3 degrees of freedom. $\Pi^A_{ij} = [Q, H]$. Thus, $$\Pi_{ij} = \Pi^S_{ij} + \Pi^A_{ij}  = \begin{pmatrix}
\Pi^S_{xx} & \Pi^S_{xy} & \Pi^S_{xz}\\
\Pi^S_{xy} & \Pi^S_{yy} & \Pi^S_{yz} \\
\Pi^S_{xz} & \Pi^S_{yz} & -\Pi^S_{xx} - \Pi^S_{yy}
\end{pmatrix} + \begin{pmatrix}
0 & \Pi^A_{xy} & \Pi^A_{xz} \\
-\Pi^A_{xy} & 0 & \Pi^A_{yz} \\
-\Pi^A_{xz} & -\Pi^A_{yz} & 0
\end{pmatrix}$$

$$\partial_j \Pi_{ij} = \begin{pmatrix}
\partial_x \Pi_{xx} + \partial_y \Pi_{xy} + \partial_z \Pi_{xz}\\
\partial_x \Pi_{yx} + \partial_y \Pi_{yy} + \partial_z \Pi_{yz}\\
\partial_x \Pi_{zx} + \partial_y \Pi_{zy} + \partial_z \Pi_{zz}
\end{pmatrix} = 
\begin{pmatrix}
\partial_x \Pi^S_{xx} + \partial_y (\Pi^S_{xy} + \Pi^A_{xy}) + \partial_z (\Pi^S_{xz} +  \Pi^A_{xz})\\
\partial_x (\Pi^S_xy - \Pi^A_{xy}) + \partial_y \Pi^S_{yy} + \partial_z (\Pi^S_{yz} + \Pi^A_{yz})\\
\partial_x (\Pi^S_xz - \Pi^A_{xz}) + \partial_y (\Pi^S_{yz} - \Pi^A_{yz}) + \partial_z (-\Pi^S_{xx} - \Pi^S_{yy})
\end{pmatrix}
$$


In [3]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_fields(dQdt, Q, H, Pi_S, Pi_A, U, A, B, C, K, Z, L, G, neighbors, points):

    for p in nb.prange(points):

        # coordinates of nearest neighbors
        xup, xdn, yup, ydn, zup, zdn = neighbors[p]
        
        # velocity gradient tensor elements. dUzdz is not needed and is not computed.
        dUdx = U[xup] - U[xdn]
        dUdy = U[yup] - U[ydn]
        dUxdz = U[zup, 0] - U[zdn, 0]
        dUydz = U[zup, 1] - U[zdn, 1]

        # vorticity tensor elements
        Wxy = (dUdx[1] - dUdy[0]) * 0.25
        Wxz = (dUdx[2] - dUxdz)   * 0.25
        Wyz = (dUdy[2] - dUydz)   * 0.25

        # \nabla^2 Q
        lap_Q = (Q[xup] + Q[xdn] + Q[yup] +
                 Q[ydn] + Q[zup] + Q[zdn] 
                             - 6 * Q[p])
        
        Qxx, Qxy, Qxz, Qyy, Qyz = Q[p, 0], Q[p, 1], Q[p, 2], Q[p, 3], Q[p, 4] 
        
        # Tr[Q^2]
        TrQ2 = 2 * (Qxx * Qxx + Qxy * Qxy + Qxz * Qxz + Qyy * Qyy + Qyz * Qyz + Qxx * Qyy)
        
        # LdG force: K \nabla^2 Q - Q (A + C Tr[Q^2]) - B (Q^2 - TrQ^2 * I/3)
        H[p] = K * lap_Q - Q[p] * (A + C * TrQ2)
        H[p, 0] -= B * (Qxx * Qxx + Qxy * Qxy + Qxz * Qxz - TrQ2 * 0.333333333)
        H[p, 1] -= B * (Qxx * Qxy + Qxy * Qyy + Qxz * Qyz)
        H[p, 2] -= B * (Qxz * Qyz - Qxz * Qyy)
        H[p, 3] -= B * (Qxy * Qxy + Qyy * Qyy + Qyz * Qyz - TrQ2 * 0.333333333)
        H[p, 4] -= B * (Qxy * Qxz - Qyz * Qxx)

        # gamma * H - (U  dot nabla) Q
        dQdt[p] = H[p] * G

        dQdt[p] -= (U[p, 0] * (Q[xup] - Q[xdn]) +
                    U[p, 1] * (Q[yup] - Q[ydn]) +
                    U[p, 2] * (Q[zup] - Q[zdn])) * 0.5
        
        # [Q, omega] + \lambda E
        dQdt[p, 0] += -2 * Qxy * Wxy - 2 * Qxz * Wxz + L * dUdx[0] * 0.5
        dQdt[p, 1] += Qxx * Wxy - Qxz * Wyz - Qyy * Wxy - Qyz * Wxz + L * (dUdy[0] + dUdx[1]) * 0.25
        dQdt[p, 2] += 2 * Qxx * Wxz + Qyy * Wxz - Qyz * Wxy + Qxy * Wyz + L * (dUxdz + dUdx[2]) * 0.25
        dQdt[p, 3] += 2 * Qxy * Wxy - 2 * Qyz * Wyz + L * dUdy[1] * 0.5
        dQdt[p, 4] += Qxx * Wyz + 2 * Qyy * Wyz + Qxz * Wxy + Qxy * Wxz + L * (dUydz + dUdy[2]) * 0.25

        Hxx, Hxy, Hxz, Hyy, Hyz = H[p, 0], H[p, 1], H[p, 2], H[p, 3], H[p, 4]

        # Pi^S = -\lambda H - \zeta Q, Pi^A = [Q, H]
        Pi_S[p] = - L * H[p] - Z * Q[p]
        Pi_A[p, 0] = Qxx * Hxy + Qxy * Hyy + Qxz * Hyz - Qxy * Hxx - Qyy * Hxy - Qyz * Hxz
        Pi_A[p, 1] = 2 * Qxx * Hxz + Qyy * Hxz - Qyz * Hxy + Qxy * Hyz - 2 * Qxz * Hxx - Qxz * Hyy
        Pi_A[p, 2] = Qxy * Hxz - 2 * Qyz * Hyy + 2 * Qyy * Hyz + Qxx * Hyz - Qyz * Hxx - Qxz * Hxy

The divergence of the stress tensor can be computed only after the stress tensor is updated at all points. Thus, we need two functions; one for $\partial_t Q$ and one for $\partial_j \Pi_{ij}$.

In [4]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_forces(dUdt, U, Pi_S, Pi_A, N, neighbors, points):
        
        for p in nb.prange(points):
            xup, xdn, yup, ydn, zup, zdn = neighbors[p]
            
            # gradient of stress tensor elemenets *TODO: eliminate un-necessary terms calculated here (like grad U tensor)*
            dPi_Sdx = Pi_S[xup] - Pi_S[xdn]
            dPi_Sdy = Pi_S[yup] - Pi_S[ydn]
            dPi_Sdz = Pi_S[zup] - Pi_S[zdn]

            dPi_Adx = Pi_A[xup] - Pi_A[xdn]
            dPi_Ady = Pi_A[yup] - Pi_A[ydn]
            dPi_Adz = Pi_A[zup] - Pi_A[zdn]
            
            # \nabla dot Pi
            dUdt[p, 0] = dPi_Sdx[0] + dPi_Sdy[1] + dPi_Ady[0] + dPi_Sdz[2] + dPi_Adz[1]
            dUdt[p, 1] = dPi_Sdx[1] - dPi_Adx[0] + dPi_Sdy[3] + dPi_Sdz[4] + dPi_Adz[2]
            dUdt[p, 2] = dPi_Sdx[2] - dPi_Adx[1] + dPi_Sdy[4] - dPi_Ady[2] - (dPi_Sdz[0] + dPi_Sdz[3])

            # - (U dot \nabla) U
            dUdt[p] -= (U[p, 0] * (U[xup] - U[xdn]) +
                        U[p, 1] * (U[yup] - U[ydn]) +
                        U[p, 2] * (U[zup] - U[zdn]))
            
            dUdt[p] *= 0.5

            # nu \nabla^2 U
            dUdt[p] += N * (U[xup] + U[xdn] + U[yup] + 
                            U[ydn] + U[zup] + U[zdn] 
                                        - 6 * U[p])    

Pressure plays the exclusive role of maintaining the second of the Navier-Stokes equations: $\partial_i U_i = 0.$

We achieve this by exploiting the linearity of divergence; if $$\partial_i U^t_i = 0,$$ and $$\partial_i \partial_t U_i = 0,$$ then $$\partial_i (U^t_i + (\partial_t U_i) dt) = 0.$$

Suppose that before we subtract the pressure gradient, $\partial_i \partial_t \tilde U_i = a$. The   $\tilde .$  indicates this is not the full time derivative, but the part of it that does not contain $\partial_i p$.

We would like $p$ such that $$\partial_i (\partial_t \tilde U_i - \partial_i p) = 0 \to a - \partial_i^2 p = 0 \to \partial_i^2 p = a.$$

$\partial_i \partial_t \tilde U_i = a$ can be coarsely approximated to leading order in $Q_{ij}$ as $-\zeta \partial_i \partial_j Q_{ij} \propto \partial_i (n_i \partial_j n_j - \epsilon_{ijk}\epsilon_{klm} n_j\partial_l n_m)$, which is known as the splay-bend parameter. We have no need to approximate it, but we will still refer to this scalar field as "SB".

Specifically, the source term of the Poisson equation,  $\partial_i \partial_t \tilde U_i = a$ cannot be computed in parallel with the other fields since it involves gradients of $\partial_t U_i$. The solving of the Poisson equation must also then happen after the source term is calculated, and only then can the pressure gradients finally be applied. This means the entire code can operate while sweeping through coordinates at least 4 times per timestep: one for the current fields, one to calculate the Poisson source, as many as required for pressure relaxation, and one for applying the pressure gradient.

In [5]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_source(SB, dUdt, neighbors, points):

    for p in nb.prange(points):
        xup, xdn, yup, ydn, zup, zdn = neighbors[p]

        # \nabla dot d(\tilde U)dt
        SB[p] = (   dUdt[xup, 0] - dUdt[xdn, 0] + 
                    dUdt[yup, 1] - dUdt[ydn, 1] + 
                    dUdt[zup, 2] - dUdt[zdn, 2]) * 0.5

In [6]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def poisson_inner(P, Pc, SB, neighbors, points):

    for p in nb.prange(points):
        xup, xdn, yup, ydn, zup, zdn = neighbors[p]
        xup_yup, xup_ydn, xup_zup, xup_zdn = neighbors[xup][2:]
        xdn_yup, xdn_ydn, xdn_zup, xdn_zdn = neighbors[xdn][2:]
        yup_zup, yup_zdn = neighbors[yup][4:]
        ydn_zup, ydn_zdn = neighbors[ydn][4:]

        # relax with 19pt O(h^4) stencil
        P[p] = ((Pc[xup] + Pc[xdn] + 
                 Pc[yup] + Pc[ydn] + 
                 Pc[zup] + Pc[zdn] ) * 0.083333333 +

                (Pc[xup_yup] + Pc[xup_ydn] +
                 Pc[xup_zup] + Pc[xup_zdn] +
                 Pc[xdn_yup] + Pc[xdn_ydn] +
                 Pc[xdn_zup] + Pc[xdn_zdn] +
                 Pc[yup_zup] + Pc[yup_zdn] +
                 Pc[ydn_zup] + Pc[ydn_zdn] ) * 0.041666667
                 
                - SB[p] * 0.25)

In [7]:
def poisson(P, Pc, SB, neighbors, points):

    threshold = 1e-4 * max(1, np.mean(np.abs(P)))
    diff = threshold + 1

    while diff > threshold:

        Pc[:] = P
        poisson_inner(P, Pc, SB, neighbors, points)
        diff = np.max(np.abs(P - Pc))

    P -= np.mean(P)

In [8]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def uncompress_flow(dUdt, P, neighbors, points):
    for p in nb.prange(points):
        xup, xdn, yup, ydn, zup, zdn = neighbors[p]

        dUdt[p, 0] -= (P[xup] - P[xdn]) * 0.5
        dUdt[p, 1] -= (P[yup] - P[ydn]) * 0.5
        dUdt[p, 2] -= (P[zup] - P[zdn]) * 0.5

In [9]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def randomize(Q, points):
    for p in nb.prange(points):

        phi = np.random.uniform(0, 2 * np.pi)
        cos_theta = np.random.uniform(-1, 1) # mercader projection
        sin_theta = np.sqrt(1 - cos_theta**2)

        nx = sin_theta * np.cos(phi)
        ny = sin_theta * np.sin(phi)
        nz = cos_theta

        Q[p, 0] = nx * nx - 0.333333333
        Q[p, 1] = nx * ny
        Q[p, 2] = nx * nz
        Q[p, 3] = ny * ny - 0.333333333
        Q[p, 4] = ny * nz

In [11]:
def make_boundary(boundary, Lx, Ly, Lz):
    neighbors = []
    if boundary == "periodic":
        for p in range(Lx * Ly * Lz):
            x = p // (Ly * Lz)
            y = (p % (Ly * Lz)) // Lz
            z = (p % (Ly * Lz)) % Lz

            xup = ((x + 1) % Lx) * (Ly * Lz) + y * Lz + z
            xdn = ((x - 1) % Lx) * (Ly * Lz) + y * Lz + z
            yup = x * (Ly * Lz) + ((y + 1) % Ly) * Lz + z
            ydn = x * (Ly * Lz) + ((y - 1) % Ly) * Lz + z
            zup = x * (Ly * Lz) + y * Lz + ((z + 1) % Lz)
            zdn = x * (Ly * Lz) + y * Lz + ((z - 1) % Lz)
            neighbors.append((xup, xdn, yup, ydn, zup, zdn))
            
    else:
        raise ValueError("Unknown boundary condition")
    return np.array(neighbors)

In [ ]:
Lx = Ly = Lz = 2**7 - 1        # system size
T            = int(5e4)        # max time steps
K            = 2**14           # elastic modulus
als          = np.inf          # active length scale
ncl          = 1               # nematic coherence length
G            = 2**(-10)        # rotational diffusivity
N            = 2**9            # viscosity
L            = 1               # flow alignment
boundary     = "periodic"      # boundary condition
runname      = "quench_2"      # output directory

In [ ]:
dt = 1/(2**3*max(G*K, N))      # time step
Z  =  K / als**2               # active stress coefficient
A  = -K / ncl**2               # 1st deGennes constant
B  =  A                        # 2nd deGennes constant
C  = -2 * A                    # 3rd deGennes constant

In [ ]:
if __name__ == "__main__":

    nb.set_num_threads(os.cpu_count() - 1)

    os.system(f"mkdir -p {runname}")
    os.system(f"mkdir -p {runname}/data/")

    neighbors = make_boundary(boundary, Lx, Ly, Lz)
    points    = len(neighbors)

    Q         = np.zeros((points, 5))  # nematic order parameter
    dQdt      = np.zeros((points, 5))  # time derivative of Q
    H         = np.zeros((points, 5))  # molecular field
    Pi_S      = np.zeros((points, 5))  # symmetric stress
    Pi_A      = np.zeros((points, 3))  # antisymmetric stress
    U         = np.zeros((points, 3))  # flow
    dUdt      = np.zeros((points, 3))  # time derivative of U
    SB        = np.zeros(points)       # pressure source
    P         = np.zeros(points)       # pressure
    Pc        = np.zeros(points)       # Poisson update copy

    randomize(Q, points)

    for t in range(T):
        
        get_fields(dQdt, Q, H, Pi_S, Pi_A, U, A, B, C, K, Z, L, G, neighbors, points)
        get_forces(dUdt, U, Pi_S, Pi_A, N, neighbors, points)
        get_source(SB, dUdt, neighbors, points)
        poisson(P, Pc, SB, neighbors, points)
        uncompress_flow(dUdt, P, neighbors, points)
        
        Q += dQdt * dt
        U += dUdt * dt

        if t % 100 == 0: np.savez(f"{runname}/data/{t:10d}.npz", Q=Q, U=U)

    os.system(f"python3 plot_3D.py {runname} {Lx}")
  